In [1]:
import pickle
with open("extracted_pdf_data.pkl", "rb") as file:
    loaded_pdf_elements = pickle.load(file)

In [ ]:
loaded_pdf_elements

In [3]:
Header=[]
Footer=[]
Title=[]
NarrativeText=[]
Text=[]
ListItem=[]
img=[]
tab=[]
for element in loaded_pdf_elements:
  if "unstructured.documents.elements.Header" in str(type(element)):
            Header.append(str(element))
  elif "unstructured.documents.elements.Footer" in str(type(element)):
            Footer.append(str(element))
  elif "unstructured.documents.elements.Title" in str(type(element)):
            Title.append(str(element))
  elif "unstructured.documents.elements.NarrativeText" in str(type(element)):
            NarrativeText.append(str(element))
  elif "unstructured.documents.elements.Text" in str(type(element)):
            Text.append(str(element))
  elif "unstructured.documents.elements.ListItem" in str(type(element)):
            ListItem.append(str(element))
  elif "unstructured.documents.elements.Image" in str(type(element)):
            img.append(str(element))
  elif "unstructured.documents.elements.Table" in str(type(element)):
            tab.append(str(element))

In [ ]:
NarrativeText

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM,BartForCausalLM,AutoModel

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name,cache_dir="/home/pms/llm_project/mm_rag_esg_financial_project/saved_model")
model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_name,cache_dir="/home/pms/llm_project/mm_rag_esg_financial_project/saved_model")

In [ ]:
model_bart

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [7]:
prompt_text = """You are an assistant tasked with summarizing text for retrieval. \
    These summaries will be embedded and used to retrieve the raw text elements. \
    Give a concise summary of the text that is well optimized for retrieval.text: {element} """

In [11]:
prompt = ChatPromptTemplate.from_template(prompt_text)

In [8]:
from langchain.llms import HuggingFacePipeline
from transformers import pipeline

In [9]:
summarizer = pipeline("summarization", model=model_bart,tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=summarizer)

Device set to use cpu
/tmp/ipykernel_739923/752795570.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=summarizer)


In [18]:
from langchain_groq import ChatGroq

In [ ]:
#from transformers import pipeline

# Load summarization model


# Summarize NarrativeText
summarized_narrative = []
for text in NarrativeText:
    summary = summarizer(text,max_length=130, min_length=30, do_sample=False)
    summarized_narrative.append(summary[0]['summary_text'])

print("Summarized NarrativeText:", summarized_narrative)


In [ ]:
# import os
# import json
# config_data = json.load(open("./config.json"))
# #MISTRALAI_API_KEY = config_data["MISTRAL_API_KEY"]
# GROQ_API_KEY = config_data["GROQ_API_KEY"]
# #os.environ["MISTRALAI_API_KEY"] = MISTRALAI_API_KEY
# os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [12]:
summarize_chain = {"element": lambda x: x} | prompt | llm | StrOutputParser()

In [ ]:
text_summaries = []
text_summaries = summarize_chain.batch(NarrativeText,{"max_concurrency":5})

In [ ]:
text_summaries